In [ ]:
!pip install -q segmentation-models-pytorch albumentations --no-cache-dir

### Libraries

Dự án sử dụng hai thư viện chính:

- `segmentation-models-pytorch`: Xây dựng mô hình image segmentation bằng PyTorch.
- `albumentations`: Thực hiện data augmentation cho ảnh và mask nhằm tăng khả năng tổng quát hóa của mô hình.

#### segmentation-models-pytorch

Thư viện cung cấp sẵn nhiều kiến trúc segmentation như: U-Net, U-Net++, FPN, PSPNet, DeepLabV3, DeepLabV3+
Đồng thời hỗ trợ nhiều backbone (ResNet, EfficientNet, DenseNet, MobileNet) và pretrained weights từ ImageNet để áp dụng transfer learning.

#### Albumentations

Thư viện chuyên dùng cho data augmentation với các phép biến đổi như: Flip, Rotate, Resize, Crop, Brightness/Contrast, Blur, Noise
Đối với bài toán segmentation, các phép biến đổi luôn được áp dụng đồng thời lên **image** và **mask** để đảm bảo tính nhất quán của dữ liệu.


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

### Libraries

#### segmentation-models-pytorch

Thư viện xây dựng mô hình **image segmentation** trên nền tảng PyTorch. Trong dự án này, thư viện được sử dụng để triển khai mô hình **U-Net** với **ResNet34** làm encoder và sử dụng **Dice Loss** cho quá trình huấn luyện.

#### albumentations

Thư viện **data augmentation** và tiền xử lý ảnh. Dự án sử dụng để **Resize**, **Horizontal Flip**, **Normalize** và áp dụng các phép biến đổi đồng thời lên **image** và **mask**.

#### ToTensorV2

Thành phần của Albumentations dùng để chuyển **image** và **mask** từ định dạng NumPy sang **PyTorch Tensor**, giúp dữ liệu có thể được đưa trực tiếp vào mô hình trong quá trình huấn luyện và dự đoán.


In [ ]:
class MelanomaDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype('float32')

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask'].unsqueeze(0)

        return image, mask

In [ ]:
image_dir = '/content/drive/MyDrive/Colab Notebooks/Dataset/Seg/data/images'
mask_dir = '/content/drive/MyDrive/Colab Notebooks/Dataset/Seg/data/masks'

image_paths = sorted(glob(os.path.join(image_dir, '*.jpg')))
mask_paths = sorted(glob(os.path.join(mask_dir, '*.png')))

train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    image_paths, mask_paths, test_size=0.2, random_state=42
)

train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(),
    A.Normalize(),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(),
    ToTensorV2()
])

train_dataset = MelanomaDataset(train_imgs, train_masks, transform=train_transform)
val_dataset = MelanomaDataset(val_imgs, val_masks, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)

model = model.to(device)

In [ ]:
loss_fn = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_epoch(loader):
    model.train()
    epoch_loss = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = loss_fn(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def validate_epoch(loader):
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = loss_fn(preds, masks)
            val_loss += loss.item()
    return val_loss / len(loader)

In [ ]:
##train lần đầu
for epoch in range(10):
    train_loss = train_epoch(train_loader)
    val_loss = validate_epoch(val_loader)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

In [ ]:
##Lưu lại checkpoint
folder_path = '/content/drive/MyDrive/Colab Notebooks'
save_path = os.path.join(folder_path, 'unet_melanoma.pth')
torch.save(model.state_dict(), save_path)
print(os.listdir(folder_path))

In [ ]:
num_epochs = 5
start_epoch = 0

for epoch in range(start_epoch, start_epoch + num_epochs):
    train_loss = train_epoch(train_loader)
    val_loss = validate_epoch(val_loader)
    print(f"Epoch {epoch+1}/{start_epoch + num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Lưu mô hình sau mỗi epoch
    torch.save(model.state_dict(), f"/content/drive/MyDrive/Colab Notebooks/unet_melanoma_epoch{epoch+1}.pth")
    print(f"✅ Saved model at epoch {epoch+1}")

In [ ]:
torch.save(model.state_dict(), f"/content/drive/MyDrive/Colab Notebooks/unet_melanoma_epoch.pth")